# LTX-2.3 chunked avatar — causal latent continuation

This notebook runs the `avatar-prototype` branch on an A100 80 GB. It uses external audio, exact or soft latent-prefix continuation, causal-boundary correction, latent overlap fusion, an optional cached negative-index identity anchor, and a single final decode to `combined.mp4`.

Run cells in order. The full experiment generates the complete audio duration. The optional comparison matrix uses two chunks per variant to reduce cost.

In [ ]:
# @title 1. Inspect A100 runtime and set process environment
import os
import subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

In [ ]:
# @title 2. Clone or update the avatar-prototype branch
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yuvrajxms09/LTX-2.git"
BRANCH = "avatar-prototype"
REPO = Path("/content/LTX-2")

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()
runner = REPO / "packages/ltx-pipelines/src/ltx_pipelines/avatar/runner.py"
assert runner.exists(), runner
print("checked out:", head)
print("identity-anchor implementation requires a958656 or later")
os.chdir(REPO)
print("working directory:", Path.cwd())

In [ ]:
%%bash
# @title 3. Install the complete locked workspace environment
set -euxo pipefail
python -m pip install -q --upgrade pip uv

uv export \
  --directory /content/LTX-2 \
  --frozen \
  --all-packages \
  --all-groups \
  --no-hashes \
  --no-emit-workspace \
  --no-emit-package ltx-kernels \
  --output-file /content/LTX-2/requirements-colab-all.txt

uv pip install --system --requirement /content/LTX-2/requirements-colab-all.txt
uv pip install --system --no-deps \
  -e /content/LTX-2/packages/ltx-core \
  -e /content/LTX-2/packages/ltx-pipelines \
  -e /content/LTX-2/packages/ltx-trainer

export TORCH_CUDA_ARCH_LIST=8.0
uv pip install --system -e /content/LTX-2/packages/ltx-kernels --no-deps --no-build-isolation

python - <<'PY'
import torch
import ltx_core
import ltx_kernels
import ltx_pipelines.avatar.runner
import ltx_trainer

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name())
print("vram_gib:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("all LTX workspace packages imported")
PY

In [ ]:
# @title 4. Authenticate and download required weights
from google.colab import userdata
from huggingface_hub import hf_hub_download, login, snapshot_download
from pathlib import Path

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

model_dir = Path("/content/LTX-2/models/ltx-2.3")
gemma_dir = Path("/content/LTX-2/models/gemma-3-12b")
model_dir.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-22b-distilled-1.1.safetensors",
    local_dir=model_dir,
)
GEMMA_ROOT = snapshot_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
    local_dir=gemma_dir,
)

print("checkpoint:", CHECKPOINT_PATH)
print("gemma:", GEMMA_ROOT)

In [ ]:
# @title 5. Select inputs; upload only when the configured path is absent
from google.colab import files
from pathlib import Path

IMAGE_PATH = "/content/LTX-2/rdj_close_up (1).jpg" # @param {type:"string"}
AUDIO_PATH = "/content/LTX-2/14s_love2.mp3" # @param {type:"string"}

def upload_one(label, destination_dir):
    print(f"Upload {label}")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one {label} file")
    name, data = next(iter(uploaded.items()))
    destination = Path(destination_dir) / Path(name).name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    return str(destination)

if not Path(IMAGE_PATH).is_file():
    IMAGE_PATH = upload_one("reference image", "/content/LTX-2/inputs")
if not Path(AUDIO_PATH).is_file():
    AUDIO_PATH = upload_one("driving audio", "/content/LTX-2/inputs")

print("image:", IMAGE_PATH)
print("audio:", AUDIO_PATH)

In [ ]:
# @title 6. Write the quality-first A100 configuration
import json
from pathlib import Path

CONFIG_PATH = Path("/content/LTX-2/avatar-causal.toml")
PROMPT = "A stable front-facing conversational avatar speaks naturally to the camera. The camera and background remain static, with natural blinking, subtle head motion, and continuous facial motion." # @param {type:"string"}

CONFIG_PATH.write_text(f'''[model]
checkpoint_path = {json.dumps(str(CHECKPOINT_PATH))}
gemma_root = {json.dumps(str(GEMMA_ROOT))}
offload = "cpu"
warm_transformer = true
compile = false

[input]
image_path = {json.dumps(str(IMAGE_PATH))}
audio_path = {json.dumps(str(AUDIO_PATH))}
prompt = {json.dumps(PROMPT)}
enhance_prompt = false

[generation]
width = 512
height = 512
frame_rate = 25.0
generation_frames = 121
overlap_frames = 25
continuation_mode = "latent-prefix"
reference_strength = 1.0
overlap_strength = 1.0
identity_anchor_strength = 0.5
seed = 10
seed_stride = 1
sigmas = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0]
max_chunks = 4

[output]
directory = "/content/LTX-2/outputs/avatar-causal-placeholder"
crf = 19
preset = "veryfast"
save_conditioning_frames = false
allow_existing = false

[diagnostics]
log_level = "INFO"
jsonl_metrics = true
synchronize_cuda = true
log_denoising_steps = true
tensor_statistics = false
''', encoding="utf-8")

print(CONFIG_PATH.read_text())

In [ ]:
# @title 7. Define reproducible experiment runner and validate the base plan
import subprocess
import sys
import time
from pathlib import Path

def run_experiment(label, *, overlap_frames, overlap_strength, identity_anchor_strength, max_chunks):
    stamp = time.strftime("%Y%m%d-%H%M%S")
    output_dir = Path(f"/content/LTX-2/outputs/{label}-{stamp}")
    cmd = [
        sys.executable, "-m", "ltx_pipelines.avatar.runner",
        "--config", str(CONFIG_PATH),
        "--set", 'generation.continuation_mode="latent-prefix"',
        "--set", "generation.generation_frames=121",
        "--set", f"generation.overlap_frames={overlap_frames}",
        "--set", f"generation.overlap_strength={overlap_strength}",
        "--set", f"generation.identity_anchor_strength={identity_anchor_strength}",
        "--set", f"generation.max_chunks={max_chunks}",
        "--set", f'output.directory="{output_dir}"',
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd="/content/LTX-2", check=True)
    return output_dir

subprocess.run([
    sys.executable, "-m", "ltx_pipelines.avatar.runner",
    "--config", str(CONFIG_PATH), "--dry-run",
], cwd="/content/LTX-2", check=True)

In [ ]:
# @title 8. Run the complete four-latent + identity-anchor experiment
RUN_FULL_EXPERIMENT = True # @param {type:"boolean"}

FULL_RUN_DIR = None
if RUN_FULL_EXPERIMENT:
    FULL_RUN_DIR = run_experiment(
        "avatar-4lat-exact-anchor05",
        overlap_frames=25,
        overlap_strength=1.0,
        identity_anchor_strength=0.5,
        max_chunks=4,
    )
    print("completed:", FULL_RUN_DIR)

In [ ]:
# @title 9. Review authoritative combined video, plan, and fusion diagnostics
import json
from IPython.display import Video, display

def review_run(run_dir):
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "manifest.json").read_text())
    summary = {
        "status": manifest.get("status"),
        "audio_duration_seconds": manifest.get("audio_duration_seconds"),
        "wall_seconds": manifest.get("wall_seconds"),
        "time_to_first_chunk_seconds": manifest.get("time_to_first_chunk_seconds"),
        "combined_output": manifest.get("combined_output"),
    }
    print(json.dumps(summary, indent=2))
    for chunk in manifest.get("completed_chunks", []):
        print(json.dumps({
            "chunk": chunk["index"],
            "emitted_frames": chunk["emitted_frames"],
            "identity_anchor_enabled": chunk.get("identity_anchor_enabled"),
            "latent_fusion": chunk.get("latent_fusion"),
            "continuity": chunk.get("continuity"),
        }, indent=2))
    combined = run_dir / "combined.mp4"
    if not combined.exists():
        raise FileNotFoundError(combined)
    display(Video(str(combined), embed=True, width=512))
    return manifest

if FULL_RUN_DIR is not None:
    FULL_MANIFEST = review_run(FULL_RUN_DIR)

In [ ]:
# @title 10. Optional two-chunk comparison matrix
RUN_COMPARISON_MATRIX = False # @param {type:"boolean"}
MATRIX_MAX_CHUNKS = 2 # @param {type:"integer"}

MATRIX_RUNS = {}
if RUN_COMPARISON_MATRIX:
    variants = [
        ("3lat-exact-no-anchor", 17, 1.0, 0.0),
        ("3lat-exact-anchor05", 17, 1.0, 0.5),
        ("4lat-exact-anchor05", 25, 1.0, 0.5),
        ("4lat-soft-anchor05", 25, 0.5, 0.5),
    ]
    for label, overlap, prefix_strength, anchor_strength in variants:
        MATRIX_RUNS[label] = run_experiment(
            f"avatar-{label}",
            overlap_frames=overlap,
            overlap_strength=prefix_strength,
            identity_anchor_strength=anchor_strength,
            max_chunks=MATRIX_MAX_CHUNKS,
        )
    print(json.dumps({key: str(value) for key, value in MATRIX_RUNS.items()}, indent=2))

In [ ]:
# @title 11. Display comparison videos and compact metrics
if not MATRIX_RUNS:
    print("Set RUN_COMPARISON_MATRIX=True in Cell 10 to generate the A/B variants.")
else:
    rows = []
    for label, run_dir in MATRIX_RUNS.items():
        manifest = json.loads((run_dir / "manifest.json").read_text())
        chunks = manifest.get("completed_chunks", [])
        rows.append({
            "variant": label,
            "wall_seconds": manifest.get("wall_seconds"),
            "frames": manifest.get("combined_output", {}).get("frames"),
            "last_boundary_rmse": chunks[-1].get("continuity", {}).get("boundary_rmse") if chunks else None,
            "last_prefix_max_abs": chunks[-1].get("continuity", {}).get("latent_prefix_max_abs") if chunks else None,
        })
        print("\n", label, run_dir)
        display(Video(str(run_dir / "combined.mp4"), embed=True, width=512))

    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        print(json.dumps(rows, indent=2))

## Interpretation

- Judge `combined.mp4`, not concatenated `chunk_*.mp4` files.
- Compare identity stability in hairline, eyes, mouth shape, face contour, clothing, background, lighting, and camera framing.
- If the anchor restricts expression, reduce `identity_anchor_strength` from `0.5` to `0.25`.
- If exact overlap shows ghosting or frozen motion, compare `overlap_strength=0.5` with the same seed.
- The identity anchor improves conditioning but cannot guarantee zero long-horizon drift from a single portrait.